In [2]:
import pandas as pd
import plotly.express as px

focus_data = pd.read_csv('Data/merged_focus_crime_data.csv', parse_dates=['Date'])
category_display_map = {
    'assault': 'Assault',
    'burglary': 'Burglary',
    'drug': 'Drug Offenses',
    'fraud': 'Fraud',
    'sex offenses': 'Sex Offenses',
    'traffic': 'Traffic Violations',
    'vandalism': 'Vandalism',
    'weapons': 'Weapons Offenses',
}
original_category = focus_data['Standardized_Category']
focus_data['Standardized_Category'] = (
    original_category.astype('string').str.strip().str.lower().map(category_display_map).fillna(original_category)
)

burglary_data = focus_data[
    focus_data['Standardized_Category'].astype('string').str.strip().str.lower().eq('burglary')
].copy()

# Keep only valid district/date rows, limit to Jul 2019-Jun 2021, and exclude out-of-city incidents.
burglary_data = burglary_data.dropna(subset=['Date', 'District'])
burglary_data = burglary_data[
    burglary_data['Date'].between('2019-07-01', '2021-06-30', inclusive='both')
].copy()
burglary_data['District'] = burglary_data['District'].astype('string').str.strip()
burglary_data = burglary_data[
    ~burglary_data['District'].str.lower().eq('out of sf')
].copy()
burglary_data['Month'] = burglary_data['Date'].dt.to_period('M').dt.to_timestamp()

monthly_counts = (
    burglary_data.groupby(['Month', 'District'], as_index=False)
    .size()
    .rename(columns={'size': 'Burglary_Count'})
    .sort_values(['Month', 'District'])
)

fig = px.line(
    monthly_counts,
    x='Month',
    y='Burglary_Count',
    color='District',
    markers=True,
    title='Monthly Burglaries by District in San Francisco',
    labels={
        'Month': 'Month',
        'Burglary_Count': 'Number of Burglaries',
        'District': 'District',
    },
)

fig.update_layout(
    template='plotly_white',
    hovermode='x unified',
    legend_title_text='District',
)
fig.update_xaxes(
    range=[pd.Timestamp('2019-07-01'), pd.Timestamp('2021-06-30')],
    tick0='2019-07-01',
    dtick='M1',
    tickformat='%b\n%Y',
)

covid_start = pd.Timestamp('2020-03-01')
fig.add_vline(
    x=covid_start,
    line_dash='dash',
    line_color='darkorange',
    line_width=2,
)
fig.add_annotation(
    x=covid_start,
    y=0.97,
    yref='paper',
    text='COVID starts',
    showarrow=False,
    xanchor='right',
    xshift=-8,
    yanchor='top',
    textangle=-90,
    font={'color': 'darkorange'},
)

fig.write_html('visualizations/burglary_monthly_district_plotly.html', include_plotlyjs='cdn')
fig.show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed